In [ ]:
# Authors: Carla Amic, Niklas Weber
# Description: Jupyter notebook to calculate the signal propagation speed in snow, 
 # the snow density and the SWE from given values of radar-measured snow thickness 
 # and actual measured snow thickness.
 # Uncertainties can be specified and processed with Gaussian error propagation.
import numpy as np
import pandas as pd
from uncertainties import unumpy as up, ufloat as uf

In [ ]:
c_air = 3.0e8 #m/s
h_r = 1.2  #radar height [m]
unc_measure = 0.01 #m measurement uncertainty snowpit
unc_radar = 0.06 #m radar resolution unceratinty of snow depth

data = {
    "Measurement": [1, 2, 3],
    # 4.3.: File1(1), File2(2), T2
    # 5.3.: File1(3), File2(4), T6
    # 5.3.: File6(5), File7(6), T5
    "h_s":   np.array([uf(0.875,unc_measure), uf(1.05,unc_measure), uf(0.92,unc_measure)]),  # snow depth (snowpit measurements)
    
    "h_i_s_1": np.array([uf(1.02, unc_radar), uf(1.3113,unc_radar), uf(1.1958, unc_radar)]),  # distance snow surface to snow-ice interface without velocity correction
    "h_i_s_2": np.array([uf(1.02, unc_radar), uf(1.4267,unc_radar), uf(1.1958, unc_radar)]),  # same for second measurement at each loction
}

df = pd.DataFrame(data) # put everything into dataframe
df["h_i_s"] = (df["h_i_s_1"]+df["h_i_s_2"])/2 # mean radar-determined snow depth

# propagation speed in snow
df["c_s"] = c_air * df["h_s"] / df["h_i_s"]
#print(df["c_s"])

# Refractive index of snow
df["n_s"] = c_air / df["c_s"]

# Relative permittivity of snow
df["eps_s"] = df["n_s"]**2

# snowpack density
df["rho_s"] = ((-1.7 + up.sqrt(2.89 + (2.8 * (df["eps_s"] - 1)))) / 1.4)*1000

# SWE
df["SWE"] = df["h_s"] * df["rho_s"]

print("\nResults:")
print(f'h_s_r = {np.array(df["h_i_s"])}')
print(f'c_s = {np.array(df["c_s"])}')
print(f'epsilon_s = {np.array(df["eps_s"])}')
print(f'rho_s = {np.array(df["rho_s"])}')
print(f'SWE = {np.array(df["SWE"])}')

# Optional: Save results to a CSV file
#df.to_csv("snow_speed.csv", index=False)


Results:
h_s_r = [1.02+/-0.042426406871192854 1.369+/-0.042426406871192854
 1.1958+/-0.042426406871192854]
c_s = [257352941.17647058+/-11101181.109593382
 230094959.82468957+/-7459949.384096534
 230807827.39588562+/-8564630.063699644]
epsilon_s = [1.3588897959183672+/-0.11723418946375276
 1.699919274376417+/-0.11022676684061337
 1.6894348298676747+/-0.12538036077717452]
rho_s = [195.39137794965936+/-59.40275771030374
 358.7287400658544+/-50.05256287909195
 353.9606616799153+/-57.10671609234914]
SWE = [170.96745570595195+/-51.494217288170674
 376.66517706914715+/-51.61544331637774
 325.6438087455221+/-51.612424288318486]


In [3]:
# average of all measurement locations

print("c_s:", df["c_s"].sum()/len(df["c_s"]))
print("n_s:", df["n_s"].sum()/len(df["n_s"]))
print("eps_s:", df["eps_s"].sum()/len(df["eps_s"]))
print("rho_s:", df["rho_s"].sum()/len(df["rho_s"]))

c_s: (2.39+/-0.05)e+08
n_s: 1.256+/-0.027
eps_s: 1.58+/-0.07
rho_s: 303+/-32
